# 02 · Análisis exploratorio (EDA) de Work-Olist

Describir variables, identificar asimetría y variabilidad, reconocer faltantes y anomalías, explorar asociaciones sin confundir correlación y causalidad.

**Modalidad:** práctica guiada, código ejecutable, interpretación y ejercicios. **Datos:** nueve CSV aportados por el usuario en `data/raw/`, con el esquema relacional de Olist. Se calculan resultados sobre estos archivos, sin simular registros ni cambiar las fuentes.

## Objetivos
Caracterizar la unidad de análisis; revisar metadatos y calidad; construir tablas descriptivas; comparar media y mediana; estudiar cuartiles y asimetría; explorar relaciones bivariadas; generar preguntas, no conclusiones causales. Los conceptos vienen de las notas históricas sobre naturaleza, escalas, distribución, EDA, correlación y valores extremos.

**Datos y advertencia:** este cuaderno detecta automáticamente los CSV reales completos en `data/raw/`. Si faltan archivos, detiene la ejecución con un error descriptivo. Las conclusiones serán condicionales al archivos entregados. Los precios están en reales brasileños (BRL) y el precio de ítems excluye flete.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
import numpy as np
import pandas as pd

# Estructura real del proyecto:
# DMEMCD_2026/Python/*.ipynb y Python/olist_tools.py
# DMEMCD_2026/work-at-olist-data/datasets/*.csv
# Funciona si el kernel arranca en la raíz o en alguna subcarpeta.
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Python" / "olist_tools.py").is_file()
             and (p / "work-at-olist-data" / "datasets").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No encuentro Python/olist_tools.py y work-at-olist-data/datasets/. "
        "Abre en VS Code la carpeta DMEMCD_2026 y selecciona su kernel."
    )
PYTHON_DIR = ROOT / "Python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))
from olist_tools import (read_tables, build_order_level, build_item_level,
                        table_inventory, missing_profile, safe_minmax, safe_zscore)
TABLAS, ORIGEN = read_tables(ROOT)
print("ORIGEN:", ORIGEN)
print("Directorio raíz:", ROOT)


ORIGEN: archivos CSV suministrados por el usuario (esquema Olist)
Directorio de trabajo: /mnt/data/Work_Olist_Notebooks_2026


In [2]:
orders = build_order_level(TABLAS)
items = build_item_level(TABLAS)
print(f"Unidad de la tabla principal: pedido; {len(orders):,} pedidos.")
display(table_inventory(TABLAS))

Unidad de la tabla principal: pedido; 99,441 pedidos.


,tabla,filas,columnas,nulos_en_celdas,memoria_MiB
0,customers,99441,5,0,30.93
1,items,112650,7,0,34.06
2,orders,99441,8,4908,24.66
3,payments,103886,5,0,16.23
4,products,32951,9,2448,6.30
5,reviews,99224,7,145903,27.77
6,sellers,3095,4,0,0.72
7,translation,71,2,0,0.01


### Primera lectura verificada de la base

En los CSV aportados hay **99 441 pedidos**, **96 096 identificadores de compradores distintos**, **112 650 líneas de artículos**, y **99 224 registros de reseñas**. La cobertura de compras se extiende desde septiembre de 2016 hasta octubre de 2018; los meses de los extremos pueden estar incompletos. No se conoce aquí el diseño de muestreo de la plataforma ni la representatividad nacional.

En `orders`, los NA de `order_delivered_customer_date` suman **2 965**; no deben imputarse automáticamente porque el pedido podría no haberse entregado. Las categorías de producto tienen **610 valores ausentes** en `products`. Estas cifras se obtienen de los archivos recibidos y también pueden recalcularse con las celdas siguientes.


## 1. ¿Qué población describe esta muestra?
Se dispone de registros de una plataforma comercial. La unidad es el pedido; los datos no forman necesariamente una muestra probabilística de todo el comercio electrónico de Brasil. `customer_id` no es equivalente a un individuo único. Los códigos de estado y tipo de pago son **cualitativos nominales**; la puntuación 1–5 es **ordinal** y, aunque se promedie por conveniencia, requiere cautela.

In [3]:
variables = pd.DataFrame([
 ["order_id","Identificador","nominal","Pedido"],
 ["customer_unique_id","Identificador","nominal","Comprador anonimizado"],
 ["subtotal_productos","BRL","cuantitativa de razón","Suma de precios de ítems"],
 ["dias_hasta_entrega","días","cuantitativa de razón","Solo entregados con fechas válidas"],
 ["puntuacion_resena","1–5","ordinal","Reseña observada; puede faltar"],
 ["order_status","categoría","nominal","Estado del pedido"],
 ["customer_state","código","nominal","Estado geográfico"],
 ], columns=["variable","unidad","escala","observación"])
display(variables)

,variable,unidad,escala,observación
0,order_id,Identificador,nominal,Pedido
1,customer_unique_id,Identificador,nominal,Comprador anonimizado
2,subtotal_productos,BRL,cuantitativa de razón,Suma de precios de ítems
3,dias_hasta_entrega,días,cuantitativa de razón,Solo entregados con fechas válidas
4,puntuacion_resena,1–5,ordinal,Reseña observada; puede faltar
5,order_status,categoría,nominal,Estado del pedido
6,customer_state,código,nominal,Estado geográfico


## 2. Completitud y calidad preliminar
Primero observar el contexto de los NA. Una fecha de entrega faltante para un pedido cancelado es distinta de un precio ausente por error de captura. La presencia de datos faltantes debe cuantificarse antes de elegir tratamiento; la limpieza se realizará en el cuarto notebook.

In [4]:
perfil=missing_profile(orders)
display(perfil.head(18))
print("Pedidos duplicados por llave:",orders["order_id"].duplicated().sum())
print("Fechas de entrega negativas (revisar, no borrar automáticamente):",
      int(orders["dias_hasta_entrega"].lt(0).sum()))
print("Ítems con precio <= 0:", int(TABLAS["items"]["price"].le(0).sum()))

,variable,faltantes,porcentaje,tipo
0,entrega_tardia,2971,2.99,boolean
1,dias_hasta_entrega,2971,2.99,float64
2,dias_retraso,2971,2.99,float64
3,order_delivered_customer_date,2965,2.98,datetime64[ns]
4,order_delivered_carrier_date,1783,1.79,datetime64[ns]
5,precio_item_mediano,775,0.78,float64
6,total_flete,775,0.78,float64
7,subtotal_productos,775,0.78,float64
8,n_vendedores,775,0.78,float64
9,n_items,775,0.78,float64


Pedidos duplicados por llave: 0
Fechas de entrega negativas (revisar, no borrar automáticamente): 0
Ítems con precio <= 0: 0


## 3. Frecuencias y tablas categóricas
Para una variable nominal describimos frecuencias y proporciones. No se deben promediar códigos postales ni ordenar categorías por supuesta magnitud. Considera tanto conteos como denominadores.

In [5]:
def frecuencias(s):
    n=s.value_counts(dropna=False).rename('conteo')
    return pd.concat([n, (n/n.sum()).rename('proporcion')],axis=1)
display(frecuencias(orders['order_status']).round(3))
display(frecuencias(orders['medio_pago_inicial']).round(3))
display(frecuencias(orders['puntuacion_resena']).round(3))

,conteo,proporcion
order_status,,
delivered,96478,0.970
shipped,1107,0.011
canceled,625,0.006
unavailable,609,0.006
invoiced,314,0.003
processing,301,0.003
created,5,0.000
approved,2,0.000


,conteo,proporcion
medio_pago_inicial,,
credit_card,76504,0.769
boleto,19784,0.199
voucher,1621,0.016
debit_card,1528,0.015
not_defined,3,0.000
NaN,1,0.000


,conteo,proporcion
puntuacion_resena,,
5.0,57029,0.573
4.0,19047,0.192
1.0,11344,0.114
3.0,8120,0.082
2.0,3133,0.032
NaN,768,0.008


## 4. Centro, dispersión y posición
Media muestral $\bar{x}=\frac{1}{n}\sum_i x_i$; varianza muestral $s^2=\frac{1}{n-1}\sum_i(x_i-\bar{x})^2$. Mediana y rango intercuartílico (RIC = Q3 − Q1) resisten mejor la influencia de unos pocos valores extremos. `describe` usa convenciones de cuantiles que debemos documentar.

In [6]:
medidas = ['subtotal_productos','total_flete','n_items','dias_hasta_entrega','dias_retraso']
display(orders[medidas].describe(percentiles=[.05,.25,.5,.75,.95,.99]).round(2))
resumen=[]
for nombre in medidas:
    s=orders[nombre].dropna()
    resumen.append({'variable':nombre,'n':s.size,'media':s.mean(),'mediana':s.median(),
                    'desv_std_muestral':s.std(ddof=1),'RIC':s.quantile(.75)-s.quantile(.25),
                    'asimetria':s.skew()})
display(pd.DataFrame(resumen).round(2))

,subtotal_productos,total_flete,n_items,dias_hasta_entrega,dias_retraso
count,98666.00,98666.00,98666.00,96470.00,96470.00
mean,137.75,22.87,1.14,12.56,-11.18
std,210.65,21.83,0.54,9.55,10.18
min,0.85,0.00,1.00,0.53,-146.02
5%,19.00,7.88,1.00,3.02,-25.94
25%,45.90,13.85,1.00,6.77,-16.24
50%,86.90,17.18,1.00,10.22,-11.95
75%,149.90,24.06,1.00,15.72,-6.39
95%,399.90,55.23,2.00,29.27,3.82
99%,998.90,105.47,3.00,46.05,18.94


,variable,n,media,mediana,desv_std_muestral,RIC,asimetria
0,subtotal_productos,98666,137.75,86.90,210.65,104.00,9.73
1,total_flete,98666,22.87,17.18,21.83,10.21,11.93
2,n_items,98666,1.14,1.00,0.54,0.00,7.53
3,dias_hasta_entrega,96470,12.56,10.22,9.55,8.95,3.83
4,dias_retraso,96470,-11.18,-11.95,10.18,9.85,2.01


**Pregunta interpretativa.** ¿Por qué el valor medio de un pedido puede diferir considerablemente del típico? Una distribución de precios comerciales suele ser asimétrica; no debemos presuponerlo sin mirar los cuantiles y una gráfica. Si aparece una observación extrema, puede ser real o un error: el diagnóstico no equivale a eliminación.

In [7]:
precio=orders['subtotal_productos'].dropna()
if not precio.empty:
    print(f"Media={precio.mean():,.2f} BRL; mediana={precio.median():,.2f} BRL; "
          f"Q95={precio.quantile(.95):,.2f} BRL; máximo={precio.max():,.2f} BRL")
    print("Interpretación: la distancia media-mediana aporta pistas sobre asimetría, no demuestra causalidad.")

Media=137.75 BRL; mediana=86.90 BRL; Q95=399.90 BRL; máximo=13,440.00 BRL
Interpretación: la distancia media-mediana aporta pistas sobre asimetría, no demuestra causalidad.


## 5. Distribución condicional y estratificación
El promedio global puede ocultar comportamientos por región, categoría, periodo o estado del pedido. La tabla siguiente separa los pedidos **entregados** y calcula medidas por estado. Cuidado: distintos tamaños y cobertura pueden afectar las comparaciones.

In [8]:
entregados = orders.loc[orders['order_status'].eq('delivered')].copy()
por_region=(entregados.groupby('customer_state',dropna=False)
    .agg(n_pedidos=('order_id','nunique'),mediana_precio=('subtotal_productos','median'),
         media_entrega_dias=('dias_hasta_entrega','mean'),
         mediana_entrega_dias=('dias_hasta_entrega','median'),
         porcentaje_retrasos=('entrega_tardia','mean'))
    .sort_values('n_pedidos',ascending=False))
display(por_region.head(9).round(3))

,n_pedidos,mediana_precio,media_entrega_dias,mediana_entrega_dias,porcentaje_retrasos
customer_state,,,,,
SP,40501,79.500,8.761,7.210,0.059
RJ,12350,89.990,15.309,12.042,0.135
MG,11354,89.900,12.009,10.314,0.056
RS,5345,89.000,15.300,13.177,0.071
PR,4923,84.990,11.992,10.426,0.05
SC,3546,89.900,14.955,13.014,0.098
BA,3256,93.250,19.335,16.914,0.14
DF,2080,88.945,12.968,11.365,0.071
ES,1995,89.000,15.789,13.637,0.122


## 6. Calendario de compras y denominadores
Los conteos mensuales dependen del intervalo cubierto por los datos. Un mes incompleto no debe compararse sin advertencia con uno completo. Los agregados de ingresos deben especificar la definición: aquí **precio de ítems** excluye flete y se basa en la tabla de pedidos agregada.

In [9]:
mensual=(orders.dropna(subset=['mes_compra']).groupby('mes_compra')
         .agg(pedidos=('order_id','nunique'),valor_productos=('subtotal_productos','sum'),
              mediana_valor=('subtotal_productos','median')).sort_index())
display(mensual.head(12).round(2))

,pedidos,valor_productos,mediana_valor
mes_compra,,,
2016-09,4,267.36,72.89
2016-10,324,49507.66,89.90
2016-12,1,10.90,10.90
2017-01,800,120312.87,87.99
2017-02,1780,247303.02,84.98
2017-03,2682,374344.30,84.94
2017-04,2404,359927.23,89.00
2017-05,3700,506071.14,87.25
2017-06,3245,433038.60,79.90


## 7. Asociación: Pearson y Spearman
Pearson cuantifica asociación **lineal**; Spearman usa rangos y puede reflejar asociaciones monótonas no necesariamente lineales. Ninguno prueba causalidad. Revisión adicional: dispersión, valores extremos, tamaño muestral, agrupación por segmentos y sesgos de selección.

In [10]:
var_num = ['subtotal_productos','total_flete','n_items','dias_hasta_entrega','dias_retraso','puntuacion_resena']
display(orders[var_num].corr(method='pearson',numeric_only=True).round(2))
display(orders[var_num].corr(method='spearman',numeric_only=True).round(2))
casos=orders[['dias_retraso','puntuacion_resena']].dropna()
if len(casos)>2:
    print('n pares observados:',len(casos),
          'Pearson:',round(casos.corr(method='pearson').iloc[0,1],3),
          'Spearman:',round(casos.corr(method='spearman').iloc[0,1],3))

,subtotal_productos,total_flete,n_items,dias_hasta_entrega,dias_retraso,puntuacion_resena
subtotal_productos,1.00,0.41,0.15,0.06,-0.01,-0.04
total_flete,0.41,1.00,0.43,0.17,-0.05,-0.09
n_items,0.15,0.43,1.00,-0.02,-0.03,-0.12
dias_hasta_entrega,0.06,0.17,-0.02,1.00,0.61,-0.33
dias_retraso,-0.01,-0.05,-0.03,0.61,1.00,-0.27
puntuacion_resena,-0.04,-0.09,-0.12,-0.33,-0.27,1.00


,subtotal_productos,total_flete,n_items,dias_hasta_entrega,dias_retraso,puntuacion_resena
subtotal_productos,1.00,0.47,0.18,0.10,-0.04,-0.03
total_flete,0.47,1.00,0.38,0.38,-0.15,-0.09
n_items,0.18,0.38,1.00,-0.03,-0.04,-0.10
dias_hasta_entrega,0.10,0.38,-0.03,1.00,0.31,-0.23
dias_retraso,-0.04,-0.15,-0.04,0.31,1.00,-0.18
puntuacion_resena,-0.03,-0.09,-0.10,-0.23,-0.18,1.00


n pares observados: 95824 Pearson: -0.266 Spearman: -0.175


### Sensibilidad a un extremo: ejemplo matemático controlado
Usamos cinco puntos **inventados** para mostrar que una observación influyente puede cambiar el coeficiente de Pearson. No atribuimos este fenómeno a Olist sin diagnosticarlo en sus tablas reales.

**Este único ejemplo matemático es deliberadamente hipotético**, no un registro agregado a los CSV Olist. Se utiliza exclusivamente para mostrar por qué una correlación puede cambiar al introducir una observación influyente.

In [11]:
ejemplo=pd.DataFrame({'x':[1,2,3,4,5,6], 'y':[1.3,2.0,3.1,4.0,5.2,5.9]})
con_extremo=pd.concat([ejemplo,pd.DataFrame({'x':[12],'y':[-9]})],ignore_index=True)
print('r sin extremo:',round(ejemplo.corr().loc['x','y'],3))
print('r con extremo:',round(con_extremo.corr().loc['x','y'],3))

r sin extremo: 0.998
r con extremo: -0.682


## 8. Valores atípicos: criterio de IQR como señal, no sentencia
El intervalo $[Q_1-1.5 RIC,\ Q_3+1.5 RIC]$ proporciona una regla exploratoria. En precios muy asimétricos puede marcar muchos valores legítimos. **No eliminaremos** observaciones en este notebook; registraremos su proporción y su contexto.

In [12]:
s=items['price'].dropna()
q1,q3=s.quantile([.25,.75]); ric=q3-q1
limite_inferior,limite_superior=q1-1.5*ric,q3+1.5*ric
bandera=(items['price']<limite_inferior)|(items['price']>limite_superior)
print(f'Límites exploratorios: [{limite_inferior:.2f}, {limite_superior:.2f}] BRL')
print('Marcados:',int(bandera.sum()),'de',int(items['price'].notna().sum()),'ítems con precio observado')
display(items.loc[bandera,['order_id','price','freight_value','categoria_mostrada']].head())

Límites exploratorios: [-102.60, 277.40] BRL
Marcados: 8427 de 112650 ítems con precio observado


,order_id,price,freight_value,categoria_mostrada
7,000576fe39319847cbb9d288c5617fa6,810.0,70.75,garden_tools
16,0009c9a17f916a706d71784483a5d643,639.0,11.34,consoles_games
26,0011d82c4b53e22e84023405fb467e57,289.0,26.33,luggage_accessories
30,00137e170939bba5a3134e2386413108,397.0,24.65,agro_industry_and_commerce
39,0017afd5076e074a48f1f1a4c7bac9c5,809.1,44.29,computers_accessories


## 9. EDA de categorías: la unidad vuelve a importar
El catálogo se une a **ítems**, no directamente a pedidos, porque un pedido puede contener varias categorías. Por tanto, las cifras siguientes son por línea de producto y no por comprador. Si quieres el número de pedidos por categoría, cuenta `order_id` únicos después de filtrar la categoría.

In [13]:
categorias=(items.groupby('categoria_mostrada',dropna=False)
    .agg(lineas=('order_id','size'),pedidos=('order_id','nunique'),
         valor_productos=('price','sum'),precio_mediano=('price','median'))
    .sort_values('lineas',ascending=False))
display(categorias.head(12).round(2))

,lineas,pedidos,valor_productos,precio_mediano
categoria_mostrada,,,,
bed_bath_table,11115,9417,1036988.68,79.05
health_beauty,9670,8836,1258681.34,79.90
sports_leisure,8641,7720,988048.97,78.00
furniture_decor,8334,6449,729762.49,65.49
computers_accessories,7827,6689,911954.32,81.99
housewares,6964,5884,632248.66,59.80
watches_gifts,5991,5624,1205005.68,129.00
telephony,4545,4199,323667.53,29.99
garden_tools,4347,3518,485256.46,59.90


### Interpretación guiada

Los resúmenes numéricos no sustituyen la comprensión de la unidad observacional. Un precio alto puede ser una venta genuina, un pedido puede no tener reseña y un pedido cancelado puede carecer de fecha de entrega por diseño. Para proponer una intervención, primero responde: **¿qué representa la columna, quién pudo registrar el dato y qué sesgo introduciría cambiarlo?**


## 10. Bitácora de hallazgos y límites
Escribe un párrafo por cada hallazgo con su indicador, su población (pedidos, ítems, compradores), su periodo y una limitación. Por ejemplo: «En los pedidos entregados y con fechas completas, la mediana de duración es ..., pero las filas sin fecha no se tratan como entrega instantánea».

**Ejercicios:**
- **Esencial:** mediana, RIC y porcentaje de NA para tres variables; muestra sus unidades.
- **Interpretación:** ¿por qué comparar las medias de dos estados sin controlar tamaño de muestra puede inducir errores?
- **Extensión:** contrasta correlación de Pearson y Spearman de `dias_retraso` con puntuación, con y sin un percentil extremo, **sin borrar los datos originales**.

**Referencias:** presentaciones históricas (análisis estadístico, EDA, valores faltantes); matriz docente; [pandas](https://pandas.pydata.org/docs/); [Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

In [14]:
# Ejercicios: escriba aquí sus tres análisis. Analiza los resultados observados en los CSV suministrados.